<a href="https://colab.research.google.com/github/gundalarakesh262-cpu/nifty100-financial-intelligence/blob/main/notebooks/Ratio_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import sqlite3

In [4]:
conn = sqlite3.connect("nifty100.db")

In [5]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,companies
1,profitandloss
2,balancesheet
3,cashflow
4,market_cap


### Loading data from provided HTML files and inserting into SQLite database

In [6]:
companies = pd.read_csv('/content/companies_cleaned.csv')
profit = pd.read_csv('/content/profitandloss_cleaned.csv')
balance = pd.read_csv('/content/balancesheet_cleaned.csv')
cashflow = pd.read_csv('/content/cashflow_cleaned.csv')
market = pd.read_csv('/content/market_cap_cleaned.csv')

In [9]:
# Insert DataFrames into SQLite
companies.to_sql('companies', conn, if_exists='replace', index=False)
profit.to_sql('profitandloss', conn, if_exists='replace', index=False)
balance.to_sql('balancesheet', conn, if_exists='replace', index=False)
cashflow.to_sql('cashflow', conn, if_exists='replace', index=False)
market.to_sql('market_cap', conn, if_exists='replace', index=False)

print("Tables created in nifty100.db from HTML files.")

# Verify that tables now exist
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

Tables created in nifty100.db from HTML files.


,name
0,companies
1,profitandloss
2,balancesheet
3,cashflow
4,market_cap


In [10]:
companies = pd.read_sql("SELECT * FROM companies", conn)
profit = pd.read_sql("SELECT * FROM profitandloss", conn)
balance = pd.read_sql("SELECT * FROM balancesheet", conn)
cashflow = pd.read_sql("SELECT * FROM cashflow", conn)
market = pd.read_sql("SELECT * FROM market_cap", conn)

The tables have been loaded into the SQLite database. Now the original data loading cell should work.

In [11]:
companies.head()
profit.head()
balance.head()
cashflow.head()

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0


In [12]:
print(profit.columns)
print(balance.columns)
print(cashflow.columns)

Index(['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout'],
      dtype='object')
Index(['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets'],
      dtype='object')
Index(['id', 'company_id', 'year', 'operating_activity', 'investing_activity',
       'financing_activity', 'net_cash_flow'],
      dtype='object')


In [13]:
import re
import numpy as np
import pandas as pd


def safe_divide(numerator, denominator):
    """
    Safely divide two columns.
    If denominator is 0 or missing, return NaN instead of crashing.
    """
    return np.where(
        (denominator == 0) | pd.isna(denominator),
        np.nan,
        numerator / denominator
    )


def extract_year(year_value):
    """
    Converts values like 'Mar-23' into 2023.
    """
    year_str = str(year_value)
    match = re.search(r"(\d{2,4})", year_str)

    if not match:
        return np.nan

    year = int(match.group(1))

    if year < 100:
        return 2000 + year

    return year


In [19]:
key_cols = ["company_id", "year", "fiscal_year"]

print("Before removing duplicates:")
print("profit:", profit.duplicated(key_cols).sum())
print("balance:", balance.duplicated(key_cols).sum())
print("cashflow:", cashflow.duplicated(key_cols).sum())

profit = profit.drop_duplicates(subset=key_cols, keep="first")
balance = balance.drop_duplicates(subset=key_cols, keep="first")
cashflow = cashflow.drop_duplicates(subset=key_cols, keep="first")

print("After removing duplicates:")
print("profit:", profit.duplicated(key_cols).sum())
print("balance:", balance.duplicated(key_cols).sum())
print("cashflow:", cashflow.duplicated(key_cols).sum())

Before removing duplicates:
profit: 13
balance: 87
cashflow: 23
After removing duplicates:
profit: 0
balance: 0
cashflow: 0


In [34]:
# Create fiscal_year column
profit["fiscal_year"] = profit["year"].apply(extract_year)
balance["fiscal_year"] = balance["year"].apply(extract_year)
cashflow["fiscal_year"] = cashflow["year"].apply(extract_year)

# Remove TTM / invalid year rows
profit2 = profit[profit["fiscal_year"].notna()].copy()
balance2 = balance[balance["fiscal_year"].notna()].copy()
cashflow2 = cashflow[cashflow["fiscal_year"].notna()].copy()

# Convert fiscal_year to integer
profit2["fiscal_year"] = profit2["fiscal_year"].astype(int)
balance2["fiscal_year"] = balance2["fiscal_year"].astype(int)
cashflow2["fiscal_year"] = cashflow2["fiscal_year"].astype(int)

# Remove duplicates using company_id + fiscal_year
key_cols = ["company_id", "fiscal_year"]

print("Before removing duplicates:")
print("profit:", profit2.duplicated(key_cols).sum())
print("balance:", balance2.duplicated(key_cols).sum())
print("cashflow:", cashflow2.duplicated(key_cols).sum())

profit2 = profit2.drop_duplicates(subset=key_cols, keep="first")
balance2 = balance2.drop_duplicates(subset=key_cols, keep="first")
cashflow2 = cashflow2.drop_duplicates(subset=key_cols, keep="first")

companies2 = companies.drop_duplicates(subset=["id"], keep="first").copy()

market2 = market.copy()
market2 = market2.rename(columns={"year": "fiscal_year"})
market2["fiscal_year"] = market2["fiscal_year"].astype(int)
market2 = market2.drop_duplicates(subset=["company_id", "fiscal_year"], keep="first")

print("After removing duplicates:")
print("profit:", profit2.duplicated(key_cols).sum())
print("balance:", balance2.duplicated(key_cols).sum())
print("cashflow:", cashflow2.duplicated(key_cols).sum())

# Drop year from balance/cashflow because we will keep profit's year label
balance2 = balance2.drop(columns=["year"], errors="ignore")
cashflow2 = cashflow2.drop(columns=["year"], errors="ignore")

# Merge main financial statements using company_id + fiscal_year
df = (
    profit2
    .merge(balance2, on=["company_id", "fiscal_year"], how="inner", suffixes=("_pl", "_bs"))
    .merge(cashflow2, on=["company_id", "fiscal_year"], how="left")
)

# Merge company data for face value
df = df.merge(
    companies2[["id", "face_value"]],
    left_on="company_id",
    right_on="id",
    how="left"
)

# Merge market cap / valuation data
df = df.merge(
    market2,
    on=["company_id", "fiscal_year"],
    how="left",
    suffixes=("", "_market")
)

print("Merged df shape:", df.shape)
print("Duplicate company-year rows after merge:", df.duplicated(["company_id", "fiscal_year"]).sum())

df.head()

Before removing duplicates:
profit: 0
balance: 88
cashflow: 12
After removing duplicates:
profit: 0
balance: 0
cashflow: 0
Merged df shape: (1137, 41)
Duplicate company-year rows after merge: 0


,id_pl,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,net_cash_flow,id_y,face_value,id,market_cap_crore,enterprise_value_crore,pe_ratio,pb_ratio,ev_ebitda,dividend_yield_pct
0,61,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,...,1.0,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,62,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,...,-31.0,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,63,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,...,-30.0,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,64,ABB,Mar 2016,2614,2250,365.0,14.0,50,3,14,...,91.0,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,65,ABB,Mar 2017,2903,2505,398.0,14.0,57,2,16,...,62.0,ABB,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
ratios = pd.DataFrame()

ratios["company_id"] = df["company_id"]
ratios["year"] = df["year"]
ratios["fiscal_year"] = df["fiscal_year"]

equity = df["equity_capital"] + df["reserves"]
capital_employed = equity + df["borrowings"]
ebit = df["operating_profit"] - df["depreciation"]
ebitda = df["operating_profit"]

# -----------------------------
# 1. Profitability KPIs
# -----------------------------
ratios["net_profit_margin_pct"] = safe_divide(df["net_profit"], df["sales"]) * 100
ratios["operating_profit_margin_pct"] = safe_divide(df["operating_profit"], df["sales"]) * 100
ratios["pbt_margin_pct"] = safe_divide(df["profit_before_tax"], df["sales"]) * 100
ratios["ebit_margin_pct"] = safe_divide(ebit, df["sales"]) * 100
ratios["expense_ratio_pct"] = safe_divide(df["expenses"], df["sales"]) * 100
ratios["return_on_equity_pct"] = safe_divide(df["net_profit"], equity) * 100
ratios["return_on_capital_employed_pct"] = safe_divide(ebit, capital_employed) * 100
ratios["return_on_assets_pct"] = safe_divide(df["net_profit"], df["total_assets"]) * 100

# -----------------------------
# 2. Debt / Leverage KPIs
# -----------------------------
ratios["debt_to_equity"] = safe_divide(df["borrowings"], equity)
ratios["debt_to_assets"] = safe_divide(df["borrowings"], df["total_assets"])
ratios["debt_to_capital"] = safe_divide(df["borrowings"], capital_employed)
ratios["liabilities_to_assets"] = safe_divide(df["total_liabilities"], df["total_assets"])
ratios["interest_coverage"] = safe_divide(
    df["operating_profit"] + df["other_income"],
    df["interest"]
)
ratios["net_debt_cr"] = df["borrowings"] - df["investments"]
ratios["net_debt_to_ebitda"] = safe_divide(ratios["net_debt_cr"], ebitda)

# -----------------------------
# 3. Cash Flow KPIs
# -----------------------------
ratios["cash_from_operations_cr"] = df["operating_activity"]
ratios["free_cash_flow_cr"] = df["operating_activity"] + df["investing_activity"]
ratios["capex_cr"] = df["investing_activity"].abs()
ratios["cfo_margin_pct"] = safe_divide(df["operating_activity"], df["sales"]) * 100
ratios["fcf_margin_pct"] = safe_divide(ratios["free_cash_flow_cr"], df["sales"]) * 100
ratios["cfo_to_pat"] = safe_divide(df["operating_activity"], df["net_profit"])
ratios["capex_to_sales_pct"] = safe_divide(ratios["capex_cr"], df["sales"]) * 100
ratios["fcf_to_ebitda"] = safe_divide(ratios["free_cash_flow_cr"], ebitda)

# -----------------------------
# 4. Per Share KPIs
# -----------------------------
share_count = safe_divide(df["equity_capital"], df["face_value"])

ratios["earnings_per_share"] = df["eps"]
ratios["book_value_per_share"] = safe_divide(equity, share_count)
ratios["dividend_payout_ratio_pct"] = df["dividend_payout"]

# -----------------------------
# 5. Valuation KPIs
# -----------------------------
ratios["market_cap_crore"] = df["market_cap_crore"]
ratios["enterprise_value_crore"] = df["enterprise_value_crore"]
ratios["pe_ratio"] = df["pe_ratio"]
ratios["pb_ratio"] = df["pb_ratio"]
ratios["ev_ebitda"] = df["ev_ebitda"]
ratios["dividend_yield_pct"] = df["dividend_yield_pct"]
ratios["earnings_yield_pct"] = safe_divide(1, df["pe_ratio"]) * 100
ratios["fcf_yield_pct"] = safe_divide(ratios["free_cash_flow_cr"], df["market_cap_crore"]) * 100

ratios.head()



,company_id,year,fiscal_year,net_profit_margin_pct,operating_profit_margin_pct,pbt_margin_pct,ebit_margin_pct,expense_ratio_pct,return_on_equity_pct,return_on_capital_employed_pct,...,book_value_per_share,dividend_payout_ratio_pct,market_cap_crore,enterprise_value_crore,pe_ratio,pb_ratio,ev_ebitda,dividend_yield_pct,earnings_yield_pct,fcf_yield_pct
0,ABB,Dec 2012,2012,8.771930,12.220206,13.006655,11.070780,87.779794,22.411128,28.284389,...,308.095238,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABB,Mar 2014,2014,8.699473,11.731107,12.961336,10.764499,88.268893,25.126904,31.091371,...,375.238095,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABB,Mar 2015,2015,10.004369,13.630406,15.028397,12.975098,86.369594,24.439701,31.696905,...,446.190476,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABB,Mar 2016,2016,9.755164,13.963275,15.225708,13.427697,86.074981,21.338912,29.372385,...,569.047619,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABB,Mar 2017,2017,9.541853,13.709955,15.018946,13.158801,86.290045,19.971161,27.541456,...,660.476190,31.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
print("Shape:", ratios.shape)

print("Duplicate company-year rows:")
print(ratios.duplicated(["company_id", "year"]).sum())

print("Missing values:")
print(ratios.isna().sum().sort_values(ascending=False).head(15))

ratios.head()

Shape: (1137, 37)
Duplicate company-year rows:
0
Missing values:
fcf_yield_pct             602
market_cap_crore          596
pe_ratio                  596
pb_ratio                  596
dividend_yield_pct        596
earnings_yield_pct        596
ev_ebitda                 596
enterprise_value_crore    596
book_value_per_share       91
interest_coverage          43
fcf_to_ebitda              31
cfo_margin_pct             19
fcf_margin_pct             19
capex_to_sales_pct         19
cfo_to_pat                 19
dtype: int64


,company_id,year,fiscal_year,net_profit_margin_pct,operating_profit_margin_pct,pbt_margin_pct,ebit_margin_pct,expense_ratio_pct,return_on_equity_pct,return_on_capital_employed_pct,...,book_value_per_share,dividend_payout_ratio_pct,market_cap_crore,enterprise_value_crore,pe_ratio,pb_ratio,ev_ebitda,dividend_yield_pct,earnings_yield_pct,fcf_yield_pct
0,ABB,Dec 2012,2012,8.771930,12.220206,13.006655,11.070780,87.779794,22.411128,28.284389,...,308.095238,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABB,Mar 2014,2014,8.699473,11.731107,12.961336,10.764499,88.268893,25.126904,31.091371,...,375.238095,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABB,Mar 2015,2015,10.004369,13.630406,15.028397,12.975098,86.369594,24.439701,31.696905,...,446.190476,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABB,Mar 2016,2016,9.755164,13.963275,15.225708,13.427697,86.074981,21.338912,29.372385,...,569.047619,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABB,Mar 2017,2017,9.541853,13.709955,15.018946,13.158801,86.290045,19.971161,27.541456,...,660.476190,31.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
import os

# Create the directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save to CSV
ratios.to_csv("../data/processed/financial_ratios_generated.csv", index=False)

# Save to SQLite table
ratios.to_sql("financial_ratios_generated", conn, if_exists="replace", index=False)

print("Ratio Engine Version 1 completed!")
print("Rows:", len(ratios))
print("Columns:", len(ratios.columns))

Ratio Engine Version 1 completed!
Rows: 1137
Columns: 37


In [38]:
print("Profit rows:", len(profit))
print("Balance rows:", len(balance))
print("Cashflow rows:", len(cashflow))
print("Merged rows:", len(df))
print("Ratios rows:", len(ratios))

print("\nUnique company-year counts:")
print("Profit:", profit[["company_id", "year"]].drop_duplicates().shape[0])
print("Balance:", balance[["company_id", "year"]].drop_duplicates().shape[0])
print("Cashflow:", cashflow[["company_id", "year"]].drop_duplicates().shape[0])
print("Merged df:", df[["company_id", "year"]].drop_duplicates().shape[0])

Profit rows: 1263
Balance rows: 1225
Cashflow rows: 1164
Merged rows: 1137
Ratios rows: 1137

Unique company-year counts:
Profit: 1263
Balance: 1225
Cashflow: 1164
Merged df: 1137


In [39]:
profit_keys = profit[["company_id", "year", "fiscal_year"]].drop_duplicates()
balance_keys = balance[["company_id", "year", "fiscal_year"]].drop_duplicates()

missing_in_balance = profit_keys.merge(
    balance_keys,
    on=["company_id", "year", "fiscal_year"],
    how="left",
    indicator=True
)

missing_in_balance = missing_in_balance[missing_in_balance["_merge"] == "left_only"]

print("Profit rows missing matching balance sheet:", len(missing_in_balance))
missing_in_balance.head(20)

Profit rows missing matching balance sheet: 184


,company_id,year,fiscal_year,_merge
12,ABB,TTM,NaN,left_only
24,ADANIENSOL,TTM,NaN,left_only
37,ADANIENT,TTM,NaN,left_only
46,ADANIGREEN,TTM,NaN,left_only
59,ADANIPORTS,TTM,NaN,left_only
72,ADANIPOWER,TTM,NaN,left_only
83,AMBUJACEM,Mar 2023 15,2023.0,left_only
85,AMBUJACEM,TTM,NaN,left_only
98,APOLLOHOSP,TTM,NaN,left_only
111,ASIANPAINT,TTM,NaN,left_only


In [40]:
import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Extra KPI Layer: Growth, CAGR, Flags, Capital Allocation
# Run this AFTER the existing ratios dataframe is created
# ---------------------------------------------------------

growth_base = df[[
    "company_id",
    "year",
    "fiscal_year",
    "sales",
    "net_profit",
    "eps",
    "operating_profit",
    "operating_activity",
    "investing_activity",
    "financing_activity",
    "borrowings",
    "equity_capital",
    "reserves"
]].copy()

growth_base = growth_base.sort_values(["company_id", "fiscal_year"]).reset_index(drop=True)

# -----------------------------
# 1. YoY Growth KPIs
# -----------------------------
growth_base["revenue_yoy_growth_pct"] = (
    growth_base.groupby("company_id")["sales"].pct_change() * 100
)

growth_base["pat_yoy_growth_pct"] = (
    growth_base.groupby("company_id")["net_profit"].pct_change() * 100
)

growth_base["eps_yoy_growth_pct"] = (
    growth_base.groupby("company_id")["eps"].pct_change() * 100
)

growth_base["operating_profit_yoy_growth_pct"] = (
    growth_base.groupby("company_id")["operating_profit"].pct_change() * 100
)


# -----------------------------
# 2. CAGR Helper Function
# -----------------------------
def calculate_cagr(current, previous, years):
    """
    CAGR = ((current / previous) ^ (1 / years) - 1) * 100

    If previous or current is <= 0, CAGR is not meaningful.
    So return NaN.
    """
    return np.where(
        (previous > 0) & (current > 0),
        ((current / previous) ** (1 / years) - 1) * 100,
        np.nan
    )


# -----------------------------
# 3. CAGR KPIs: 3Y, 5Y, 10Y
# -----------------------------
for period in [3, 5, 10]:
    growth_base[f"revenue_cagr_{period}y_pct"] = growth_base.groupby("company_id")["sales"].transform(
        lambda x: calculate_cagr(x, x.shift(period), period)
    )

    growth_base[f"pat_cagr_{period}y_pct"] = growth_base.groupby("company_id")["net_profit"].transform(
        lambda x: calculate_cagr(x, x.shift(period), period)
    )

    growth_base[f"eps_cagr_{period}y_pct"] = growth_base.groupby("company_id")["eps"].transform(
        lambda x: calculate_cagr(x, x.shift(period), period)
    )

    growth_base[f"operating_profit_cagr_{period}y_pct"] = growth_base.groupby("company_id")["operating_profit"].transform(
        lambda x: calculate_cagr(x, x.shift(period), period)
    )


# -----------------------------
# 4. Business Flags / Edge Case Flags
# -----------------------------
equity_extra = growth_base["equity_capital"] + growth_base["reserves"]
fcf_extra = growth_base["operating_activity"] + growth_base["investing_activity"]

growth_base["debt_free_flag"] = growth_base["borrowings"].fillna(0).eq(0)
growth_base["negative_equity_flag"] = equity_extra <= 0
growth_base["cfo_positive_flag"] = growth_base["operating_activity"] > 0
growth_base["fcf_positive_flag"] = fcf_extra > 0
growth_base["high_quality_earnings_flag"] = growth_base["operating_activity"] > growth_base["net_profit"]

# FCF negative for 3 consecutive years
growth_base["fcf_negative_flag"] = fcf_extra < 0
growth_base["fcf_negative_3yr_flag"] = (
    growth_base.groupby("company_id")["fcf_negative_flag"]
    .rolling(3)
    .sum()
    .reset_index(level=0, drop=True)
    .eq(3)
)

growth_base["revenue_growth_positive_flag"] = growth_base["revenue_yoy_growth_pct"] > 0
growth_base["pat_growth_positive_flag"] = growth_base["pat_yoy_growth_pct"] > 0


# -----------------------------
# 5. Capital Allocation Pattern
# -----------------------------
def sign_label(value):
    if pd.isna(value):
        return "missing"
    if value > 0:
        return "positive"
    if value < 0:
        return "negative"
    return "zero"


def capital_allocation_label(row):
    cfo = sign_label(row["operating_activity"])
    cfi = sign_label(row["investing_activity"])
    cff = sign_label(row["financing_activity"])

    if cfo == "positive" and cfi == "negative" and cff == "negative":
        return "Reinvesting + Returning Capital"

    if cfo == "positive" and cfi == "negative" and cff == "positive":
        return "Growth Funded by Financing"

    if cfo == "positive" and cfi == "positive" and cff == "negative":
        return "Asset Sale + Debt/Dividend Outflow"

    if cfo == "positive" and cfi == "positive" and cff == "positive":
        return "Cash Accumulation / Asset Sale"

    if cfo == "negative" and cff == "positive":
        return "Possible Distress Financing"

    if cfo == "negative" and cfi == "negative":
        return "Cash Burn + Investment"

    if cfo == "negative" and cfi == "positive":
        return "Operations Weak / Selling Assets"

    return "Mixed / Neutral"


growth_base["cfo_sign"] = growth_base["operating_activity"].apply(sign_label)
growth_base["cfi_sign"] = growth_base["investing_activity"].apply(sign_label)
growth_base["cff_sign"] = growth_base["financing_activity"].apply(sign_label)

growth_base["capital_allocation_pattern"] = growth_base.apply(capital_allocation_label, axis=1)


# -----------------------------
# 6. Merge Extra KPIs into ratios
# -----------------------------
extra_kpis = growth_base[[
    "company_id",
    "fiscal_year",
    "revenue_yoy_growth_pct",
    "pat_yoy_growth_pct",
    "eps_yoy_growth_pct",
    "operating_profit_yoy_growth_pct",
    "revenue_cagr_3y_pct",
    "pat_cagr_3y_pct",
    "eps_cagr_3y_pct",
    "operating_profit_cagr_3y_pct",
    "revenue_cagr_5y_pct",
    "pat_cagr_5y_pct",
    "eps_cagr_5y_pct",
    "operating_profit_cagr_5y_pct",
    "revenue_cagr_10y_pct",
    "pat_cagr_10y_pct",
    "eps_cagr_10y_pct",
    "operating_profit_cagr_10y_pct",
    "debt_free_flag",
    "negative_equity_flag",
    "cfo_positive_flag",
    "fcf_positive_flag",
    "high_quality_earnings_flag",
    "fcf_negative_3yr_flag",
    "revenue_growth_positive_flag",
    "pat_growth_positive_flag",
    "capital_allocation_pattern"
]]

ratios = ratios.merge(
    extra_kpis,
    on=["company_id", "fiscal_year"],
    how="left"
)

print("Extra KPI layer added!")
print("Final shape:", ratios.shape)
print("Duplicate company-year rows:", ratios.duplicated(["company_id", "fiscal_year"]).sum())

ratios.head()

/tmp/ipykernel_8579/787727226.py:40: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  growth_base.groupby("company_id")["eps"].pct_change() * 100
/tmp/ipykernel_8579/787727226.py:44: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  growth_base.groupby("company_id")["operating_profit"].pct_change() * 100


Extra KPI layer added!
Final shape: (1137, 62)
Duplicate company-year rows: 0


,company_id,year,fiscal_year,net_profit_margin_pct,operating_profit_margin_pct,pbt_margin_pct,ebit_margin_pct,expense_ratio_pct,return_on_equity_pct,return_on_capital_employed_pct,...,operating_profit_cagr_10y_pct,debt_free_flag,negative_equity_flag,cfo_positive_flag,fcf_positive_flag,high_quality_earnings_flag,fcf_negative_3yr_flag,revenue_growth_positive_flag,pat_growth_positive_flag,capital_allocation_pattern
0,ABB,Dec 2012,2012,8.771930,12.220206,13.006655,11.070780,87.779794,22.411128,28.284389,...,NaN,True,False,True,True,False,False,False,False,Reinvesting + Returning Capital
1,ABB,Mar 2014,2014,8.699473,11.731107,12.961336,10.764499,88.268893,25.126904,31.091371,...,NaN,True,False,True,True,False,False,True,True,Reinvesting + Returning Capital
2,ABB,Mar 2015,2015,10.004369,13.630406,15.028397,12.975098,86.369594,24.439701,31.696905,...,NaN,True,False,True,True,False,False,True,True,Reinvesting + Returning Capital
3,ABB,Mar 2016,2016,9.755164,13.963275,15.225708,13.427697,86.074981,21.338912,29.372385,...,NaN,True,False,True,True,False,False,True,True,Reinvesting + Returning Capital
4,ABB,Mar 2017,2017,9.541853,13.709955,15.018946,13.158801,86.290045,19.971161,27.541456,...,NaN,True,False,True,True,True,False,True,True,Reinvesting + Returning Capital


In [43]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../logs", exist_ok=True)

# Save final ratios output
ratios.to_csv("../data/processed/financial_ratios_generated.csv", index=False)

# Save to SQLite
ratios.to_sql("financial_ratios_generated", conn, if_exists="replace", index=False)

# Save capital allocation file separately
capital_allocation = growth_base[[
    "company_id",
    "year",
    "fiscal_year",
    "cfo_sign",
    "cfi_sign",
    "cff_sign",
    "capital_allocation_pattern"
]]

capital_allocation.to_csv("../data/processed/capital_allocation.csv", index=False)

print("Ratio Engine Final Version completed!")
print("Rows:", len(ratios))
print("Columns:", len(ratios.columns))

Ratio Engine Final Version completed!
Rows: 1137
Columns: 62


In [42]:
print("Final shape:", ratios.shape)
print("Duplicate company-year rows:", ratios.duplicated(["company_id", "fiscal_year"]).sum())

print("\nColumn count:", len(ratios.columns))

print("\nColumns:")
for col in ratios.columns:
    print(col)

print("\nMissing values top 20:")
print(ratios.isna().sum().sort_values(ascending=False).head(20))

Final shape: (1137, 62)
Duplicate company-year rows: 0

Column count: 62

Columns:
company_id
year
fiscal_year
net_profit_margin_pct
operating_profit_margin_pct
pbt_margin_pct
ebit_margin_pct
expense_ratio_pct
return_on_equity_pct
return_on_capital_employed_pct
return_on_assets_pct
debt_to_equity
debt_to_assets
debt_to_capital
liabilities_to_assets
interest_coverage
net_debt_cr
net_debt_to_ebitda
cash_from_operations_cr
free_cash_flow_cr
capex_cr
cfo_margin_pct
fcf_margin_pct
cfo_to_pat
capex_to_sales_pct
fcf_to_ebitda
earnings_per_share
book_value_per_share
dividend_payout_ratio_pct
market_cap_crore
enterprise_value_crore
pe_ratio
pb_ratio
ev_ebitda
dividend_yield_pct
earnings_yield_pct
fcf_yield_pct
revenue_yoy_growth_pct
pat_yoy_growth_pct
eps_yoy_growth_pct
operating_profit_yoy_growth_pct
revenue_cagr_3y_pct
pat_cagr_3y_pct
eps_cagr_3y_pct
operating_profit_cagr_3y_pct
revenue_cagr_5y_pct
pat_cagr_5y_pct
eps_cagr_5y_pct
operating_profit_cagr_5y_pct
revenue_cagr_10y_pct
pat_cagr_10y_

## Financial Ratio Engine — Module 2

This module calculates financial KPIs for Nifty 100 companies using Profit & Loss, Balance Sheet, Cash Flow, Companies, and Market Cap datasets.

### Input Tables Used
- `profitandloss`
- `balancesheet`
- `cashflow`
- `companies`
- `market_cap`

### Join Logic
The financial statements are merged using:
- `company_id`
- `fiscal_year`

The original `year` field is preserved for display, but `fiscal_year` is used for safer joins because some source year labels are inconsistent.

### KPI Categories Generated
The Ratio Engine generates KPIs across the following categories:

1. Profitability ratios  
2. Return ratios  
3. Debt and leverage ratios  
4. Cash flow ratios  
5. Per-share metrics  
6. Valuation ratios  
7. YoY growth metrics  
8. 3Y, 5Y, and 10Y CAGR metrics  
9. Business flags  
10. Capital allocation pattern labels  

### Edge Cases Handled
- Division by zero returns `NaN` instead of crashing.
- TTM rows are excluded from annual ratio computation.
- Duplicate company-year rows are removed before merging.
- Market-cap related ratios are missing for older years where market data is unavailable.
- Debt-free companies are flagged.
- Negative equity companies are flagged.
- Negative FCF for 3 consecutive years is flagged.
- Capital allocation pattern is classified using CFO, CFI, and CFF signs.

### Outputs
- `financial_ratios_generated.csv`
- `financial_ratios_generated` SQLite table
- `capital_allocation.csv`

In [45]:
edge_case_summary = {
    "total_rows": len(ratios),
    "total_columns": len(ratios.columns),
    "duplicate_company_year_rows": ratios.duplicated(["company_id", "fiscal_year"]).sum(),
    "debt_free_companies_rows": ratios["debt_free_flag"].sum(),
    "negative_equity_rows": ratios["negative_equity_flag"].sum(),
    "fcf_negative_3yr_rows": ratios["fcf_negative_3yr_flag"].sum(),
    "missing_market_cap_rows": ratios["market_cap_crore"].isna().sum(),
    "missing_pe_ratio_rows": ratios["pe_ratio"].isna().sum(),
    "missing_interest_coverage_rows": ratios["interest_coverage"].isna().sum()
}

edge_case_summary

{'total_rows': 1137,
 'total_columns': 62,
 'duplicate_company_year_rows': np.int64(0),
 'debt_free_companies_rows': np.int64(81),
 'negative_equity_rows': np.int64(0),
 'fcf_negative_3yr_rows': np.int64(98),
 'missing_market_cap_rows': np.int64(596),
 'missing_pe_ratio_rows': np.int64(596),
 'missing_interest_coverage_rows': np.int64(43)}

In [46]:
import os

os.makedirs("../logs", exist_ok=True)

with open("../logs/ratio_edge_cases.log", "w") as f:
    f.write("Ratio Engine Edge Case Summary\n")
    f.write("=" * 40 + "\n\n")
    for key, value in edge_case_summary.items():
        f.write(f"{key}: {value}\n")

print("ratio_edge_cases.log created successfully!")

ratio_edge_cases.log created successfully!
